<a href="https://colab.research.google.com/github/Mjeed19/Athar/blob/main/Athar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1- Data Loading & Structure

In [ ]:
#Data Handling
import pandas as pd
import numpy as np

#Visualization
import seaborn as sns
import matplotlib.pyplot as plt

#Preprocessing (encoding, scaling)
from sklearn.preprocessing import StandardScaler

#Utilities
import warnings
warnings.filterwarnings("ignore")

#Plot Style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("Libraries imported successfully and ready to use!")

Libraries imported successfully and ready to use!


In [ ]:
import kagglehub
import os

# Download latest version
path_to_dataset_dir = kagglehub.dataset_download("wcukierski/enron-email-dataset")

print("Path to dataset files:", path_to_dataset_dir)

# Construct the full path to the CSV file. Assuming the main CSV is named 'emails.csv'.
csv_file_path = os.path.join(path_to_dataset_dir, 'emails.csv')

# Read the CSV file
df = pd.read_csv(csv_file_path)

Using Colab cache for faster access to the 'enron-email-dataset' dataset.
Path to dataset files: /kaggle/input/enron-email-dataset


#data understanding


In [ ]:
df.head()

,date,sender,recipient,subject,body,file
0,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast\n\n,allen-p/_sent_mail/1.
1,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...,allen-p/_sent_mail/10.
2,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!,allen-p/_sent_mail/100.
3,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s...",allen-p/_sent_mail/1000.
4,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.,allen-p/_sent_mail/1001.


In [ ]:
df.shape

(517401, 2)

In [ ]:
df.size

1034802

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   file     517401 non-null  object
 1   message  517401 non-null  object
dtypes: object(2)
memory usage: 7.9+ MB


#Data cleaning

In [ ]:
df.isnull().sum()

,0
file,0
message,0


In [ ]:
df.duplicated().sum()

np.int64(0)

## Parsing the raw message
The `message` column is one long string per email. `email.message_from_string` reads it and gives us access to each header separately, so we can turn one text column into a proper table.

In [ ]:
import email
def parse_email(raw_text):
    msg = email.message_from_string(raw_text)
    return {
        'date': msg.get('Date'),
        'sender': msg.get('From'),
        'recipient': msg.get('To'),
        'subject': msg.get('Subject'),
        'body': msg.get_payload()
    }

In [ ]:
# Re-load the DataFrame to ensure 'file' and 'message' columns are present.
# This is necessary because `df` in the current kernel state has already been transformed
# and no longer contains 'file' or 'message' columns.
df_raw = pd.read_csv(csv_file_path)

# Now, extract the 'file' column before parsing
original_files = df_raw['file']

# Parse the email messages from the 'message' column of the raw DataFrame
df_parsed = pd.DataFrame([parse_email(m) for m in df_raw['message']])

# Add the 'file' column back to the new DataFrame
df_parsed['file'] = original_files

# Assign the new parsed DataFrame back to df
df = df_parsed
df.head()

,date,sender,recipient,subject,body,file
0,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast\n\n,allen-p/_sent_mail/1.
1,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...,allen-p/_sent_mail/10.
2,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!,allen-p/_sent_mail/100.
3,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s...",allen-p/_sent_mail/1000.
4,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.,allen-p/_sent_mail/1001.


In [ ]:
df.head()

,date,sender,recipient,subject,body,file
0,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast\n\n,allen-p/_sent_mail/1.
1,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...,allen-p/_sent_mail/10.
2,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!,allen-p/_sent_mail/100.
3,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s...",allen-p/_sent_mail/1000.
4,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.,allen-p/_sent_mail/1001.


In [ ]:
df.columns.tolist()

['date', 'sender', 'recipient', 'subject', 'body', 'file']

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   date       517401 non-null  object
 1   sender     517401 non-null  object
 2   recipient  495554 non-null  object
 3   subject    517401 non-null  object
 4   body       517401 non-null  object
 5   file       517401 non-null  object
dtypes: object(6)
memory usage: 23.7+ MB
